# Product Satisfaction from Review Text

This is an **independent experimental extension**. It does not improve the
selfie colour matcher directly. The goal is to predict whether a review is
positive from its text using a TF-IDF and logistic-regression baseline.

The split is grouped by product so that reviews for the same product do
not appear in both training and test sets.


## 1. Setup


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/YOUR_USERNAME/foundation-shade-recommender.git"


def find_project_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return None


project_root = find_project_root(Path.cwd())

if project_root is None and "google.colab" in sys.modules:
    if "YOUR_USERNAME" in REPOSITORY_URL:
        raise ValueError(
            "Replace YOUR_USERNAME in REPOSITORY_URL after publishing the project to GitHub."
        )
    project_root = Path("/content/foundation-shade-recommender")
    if not project_root.exists():
        subprocess.run(["git", "clone", REPOSITORY_URL, str(project_root)], check=True)

if project_root is None:
    raise FileNotFoundError("Run this notebook from inside the project folder.")

os.chdir(project_root)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", "."],
    check=True,
)
print("Project root:", project_root)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.metrics import ConfusionMatrixDisplay

from foundation_matcher.config import LUXXIFY_REVIEW_URL
from foundation_matcher.review_model import (
    build_review_classifier,
    evaluate_review_classifier,
    grouped_train_test_split,
    prepare_review_data,
)


## 2. Load and prepare reviews

Ratings of 4 or 5 are labelled positive. Because the dataset is strongly
imbalanced, later evaluation includes balanced accuracy, class-specific
precision/recall/F1, ROC AUC, and average precision—not accuracy alone.


In [ ]:
makeup_reviews = pd.read_csv(LUXXIFY_REVIEW_URL, low_memory=False)

print("Raw review shape:", makeup_reviews.shape)
print("Available columns:", makeup_reviews.columns.tolist())

review_ml = prepare_review_data(
    makeup_reviews,
    positive_threshold=4,
)

print("Prepared examples:", len(review_ml))
display(
    review_ml["positive_review"]
    .value_counts()
    .rename_axis("class")
    .to_frame("count")
    .assign(proportion=lambda table: table["count"] / table["count"].sum())
)


### Optional Colab-sized sample

The full dataset can be used by setting `MAX_ROWS = None`. The default
keeps the baseline practical on a standard Colab runtime while retaining
both classes. This is a development sample, not a final benchmark.


In [ ]:
MAX_ROWS = 120_000

if MAX_ROWS is not None and len(review_ml) > MAX_ROWS:
    fraction = MAX_ROWS / len(review_ml)
    review_ml = (
        review_ml.groupby("positive_review", group_keys=False)
        .sample(frac=fraction, random_state=42)
        .sample(frac=1, random_state=42)
        .reset_index(drop=True)
    )

print("Examples used:", len(review_ml))


## 3. Make a product-grouped train/test split


In [ ]:
(
    X_train,
    X_test,
    y_train,
    y_test,
    train_groups,
    test_groups,
) = grouped_train_test_split(review_ml)

product_overlap = set(train_groups).intersection(test_groups)
assert not product_overlap, "Product leakage detected."

print(f"Training reviews: {len(X_train):,}")
print(f"Testing reviews: {len(X_test):,}")
print("Products appearing in both sets:", len(product_overlap))
print("Training positive rate:", f"{y_train.mean():.1%}")
print("Testing positive rate:", f"{y_test.mean():.1%}")


## 4. Train the baseline


In [ ]:
review_classifier = build_review_classifier(
    max_features=40_000,
    minimum_document_frequency=5,
)

review_classifier.fit(X_train, y_train)
print("Training complete.")


## 5. Evaluate with imbalance-aware metrics


In [ ]:
metrics, classification_table, confusion = evaluate_review_classifier(
    review_classifier,
    X_test,
    y_test,
)

always_positive_accuracy = y_test.mean()
print("Always-positive accuracy:", f"{always_positive_accuracy:.3f}")
display(metrics.to_frame().round(3))
display(classification_table.round(3))

ConfusionMatrixDisplay(
    confusion_matrix=confusion,
    display_labels=["Negative", "Positive"],
).plot(cmap="Blues", colorbar=False)
plt.title("Review Satisfaction Confusion Matrix")
plt.grid(False)
plt.show()


## 6. Inspect influential words and phrases


In [ ]:
feature_names = review_classifier.named_steps["tfidf"].get_feature_names_out()
coefficients = review_classifier.named_steps["classifier"].coef_[0]

coefficient_table = pd.DataFrame(
    {"term": feature_names, "coefficient": coefficients}
)

print("Terms most associated with negative reviews")
display(coefficient_table.nsmallest(15, "coefficient"))

print("Terms most associated with positive reviews")
display(coefficient_table.nlargest(15, "coefficient"))


## 7. Interpretation and next steps

Compare the trained model with the always-positive baseline. High raw
accuracy is not impressive when most reviews are positive. Balanced
accuracy and negative-class recall reveal whether the classifier learns
useful minority-class signals.

Ratings are used to create the label, so this experiment predicts
rating-derived sentiment rather than long-term product satisfaction.
Stronger work could add calibration, repeated grouped cross-validation,
error analysis, and product metadata while preserving product-level
separation between training and evaluation.
